# 🚀 Phase 11: Best Model Fine-Tuning & Evaluation

## Overview
This notebook performs fine-tuning on the best model from Phase 10 with the following requirements:
- **Checkpoint**: `h5_omnifusion_medium_fold4_best.pt` (Fold 4)
- **Metrics**: Accuracy, Precision, F1-Score, Recall, AUC (for Train, Val, and Test separately).
- **Visualization**: Confusion Matrices for all three splits.
- **Data**: Uses all available data samples.

## Step 1: Setup & Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/phase2'):
    !git clone https://github.com/nithin12342/phase2.git /content/phase2
else:
    %cd /content/phase2
    !git fetch origin && git reset --hard origin/main
    %cd /content

!pip install torch torchvision torchaudio --quiet
!pip install transformers h5py pandas scikit-learn tqdm matplotlib seaborn --quiet
print("✅ Setup complete!")

## Step 2: Configuration

In [ ]:
import sys
sys.path.insert(0, '/content/phase2/ml_pipeline/h5_omnifusion')

# Paths
DATA_ROOT = "/content/drive/MyDrive/DAIC-WOZ_Datasets"
DATA_DIR = f"{DATA_ROOT}/H5_OmniFusion_Output"
LABELS = f"{DATA_DIR}/all_labels.csv"

# Robust Checkpoint Discovery
POSSIBLE_CHECKPOINT_DIRS = [
    f"{DATA_ROOT}/checkpoints_phase10_finetune",
    f"{DATA_ROOT}/checkpoints_phase10",
]

BASE_NAME = "h5_omnifusion_medium_fold4_best"
EXTENSIONS = [".pt", ".p", ".pth"]

RESUME_PATH = None
for d in POSSIBLE_CHECKPOINT_DIRS:
    for ext in EXTENSIONS:
        path = os.path.join(d, BASE_NAME + ext)
        if os.path.exists(path):
            RESUME_PATH = path
            break
    if RESUME_PATH: break

OUT_DIR = f"{DATA_ROOT}/checkpoints_phase11_best_finetune"

# Hyperparameters
TIER = "medium"
EPOCHS = 5           
LR = 2e-6            
BATCH_SIZE = 16
FOLD = 4             

!mkdir -p {OUT_DIR}
if RESUME_PATH:
    print(f"✅ Checkpoint Found: {RESUME_PATH}")
else:
    print(f"❌ ERROR: Could not find checkpoint {BASE_NAME} in {POSSIBLE_CHECKPOINT_DIRS}")
    print("   Please verify the file exists on Google Drive.")

print(f"📁 Results will be saved in: {OUT_DIR}")

## Step 3: Custom Trainer with Multi-Split Metrics

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from tqdm import tqdm

from config.model_config import H5Config, ComputeTier
from config.training_config import TrainingConfig
from src.models.h5_omnifusion import H5OmniFusion
from src.training.trainer import H5Trainer, FocalLossBinary
from src.data.h5_dataset import create_h5_dataloaders_kfold

class ComprehensiveTrainer(H5Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.history = {
            'train': {'acc': [], 'prec': [], 'rec': [], 'f1': [], 'auc': [], 'loss': []},
            'val':   {'acc': [], 'prec': [], 'rec': [], 'f1': [], 'auc': [], 'loss': []},
            'test':  {'acc': [], 'prec': [], 'rec': [], 'f1': [], 'auc': [], 'loss': []}
        }

    def train(self, save_path: str = "best_model.pt"):
        print(f"\n🚀 Starting Fine-Tuning for {self.config.n_epochs} Epochs")
        
        for epoch in range(self.current_epoch, self.config.n_epochs):
            self.current_epoch = epoch
            
            # 1. Train
            train_metrics = self._train_epoch()
            self._update_history('train', train_metrics)
            
            # 2. Validate
            val_metrics = self._evaluate(self.val_loader, desc="Validating")
            self._update_history('val', val_metrics)
            
            # 3. Test
            test_metrics = self._evaluate(self.test_loader, desc="Testing")
            self._update_history('test', test_metrics)
            
            # 4. Display Epoch Summary
            self._log_epoch_metrics(epoch, train_metrics, val_metrics, test_metrics)
            
            # 5. Checkpointing
            if val_metrics['f1'] > self.best_val_f1:
                self.best_val_f1 = val_metrics['f1']
                self._save_checkpoint(save_path)
                print(f"  ✨ New Best F1: {self.best_val_f1:.4f} (Saved)")

    def _update_history(self, split, metrics):
        self.history[split]['acc'].append(metrics.get('accuracy', 0))
        self.history[split]['prec'].append(metrics.get('precision', 0))
        self.history[split]['rec'].append(metrics.get('recall', 0))
        self.history[split]['f1'].append(metrics.get('f1', 0))
        self.history[split]['auc'].append(metrics.get('auc', 0))
        self.history[split]['loss'].append(metrics.get('loss', 0))

    def _log_epoch_metrics(self, epoch, t, v, te):
        print(f"\n📅 Epoch {epoch+1}/{self.config.n_epochs}")
        print(f"  {'Metric':<10} | {'Train':<10} | {'Val':<10} | {'Test':<10}")
        print(f"  {'-'*43}")
        for m in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
            print(f"  {m.title()[:10]:<10} | {t[m]:<10.4f} | {v[m]:<10.4f} | {te[m]:<10.4f}")
        print(f"  {'Loss':<10} | {t['loss']:<10.4f} | {v['loss']:<10.4f} | {te['loss']:<10.4f}")

    def plot_final_results(self):
        """Display curves and confusion matrices for all splits."""
        epochs = range(1, len(self.history['train']['acc']) + 1)
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        
        # 1. Training Curves (F1 Score)
        axes[0,0].plot(epochs, self.history['train']['f1'], 'b-o', label='Train F1')
        axes[0,0].plot(epochs, self.history['val']['f1'], 'g-o', label='Val F1')
        axes[0,0].plot(epochs, self.history['test']['f1'], 'r-o', label='Test F1')
        axes[0,0].set_title('F1-Score Over Epochs'); axes[0,0].legend(); axes[0,0].grid(True)
        
        # 2. Training Curves (Loss)
        axes[1,0].plot(epochs, self.history['train']['loss'], 'b-o', label='Train Loss')
        axes[1,0].plot(epochs, self.history['val']['loss'], 'g-o', label='Val Loss')
        axes[1,0].plot(epochs, self.history['test']['loss'], 'r-o', label='Test Loss')
        axes[1,0].set_title('Loss Over Epochs'); axes[1,0].legend(); axes[1,0].grid(True)
        
        # 3. Confusion Matrices
        splits = ['train', 'val', 'test']
        loaders = [self.train_loader, self.val_loader, self.test_loader]
        
        for i, (split, loader) in enumerate(zip(splits, loaders)):
            metrics = self._evaluate(loader, desc=f"Final {split} CM")
            cm = [[metrics['tn'], metrics['fp']], [metrics['fn'], metrics['tp']]]
            
            ax = axes[0 if i < 2 else 1, i+1 if i < 2 else 1]
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                        xticklabels=['Normal', 'Depressed'], 
                        yticklabels=['Normal', 'Depressed'])
            ax.set_title(f'Confusion Matrix: {split.upper()}')
            ax.set_xlabel('Predicted'); ax.set_ylabel('True')
            
        plt.tight_layout()
        plt.show()

## Step 4: Execution

In [ ]:
import random

if not RESUME_PATH:
    raise FileNotFoundError(f"Cannot proceed without checkpoint: {BASE_NAME}")

# 1. Initialize Dataset & Model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = H5Config.from_tier(ComputeTier.MEDIUM)
model = H5OmniFusion(config)

# 2. Load Checkpoint
print(f"🔄 Loading checkpoint: {RESUME_PATH}")
try:
    ckpt = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state, strict=False)
    print("✅ Checkpoint loaded successfully!")
except Exception as e:
    print(f"❌ Error loading checkpoint: {e}")
    raise e

# 3. Set up Training Config
tc = TrainingConfig()
tc.n_epochs = EPOCHS
tc.optimizer.lr = LR
tc.batch_size = BATCH_SIZE

# 4. Data Loaders (Fold 4)
print("📊 Initializing Data Loaders (this may take a minute to index Drive)...")
train_loader, val_loader, test_loader = create_h5_dataloaders_kfold(
    h5_dir=DATA_DIR, 
    labels_csv=LABELS,
    batch_size=BATCH_SIZE, 
    n_folds=5, 
    fold_idx=FOLD, 
    seed=42
)

# 5. Initialize Trainer
trainer = ComprehensiveTrainer(
    model=model, 
    train_loader=train_loader, 
    val_loader=val_loader,
    test_loader=test_loader, 
    config=tc, 
    device=device
)

# 6. Run Training
save_path = f"{OUT_DIR}/h5_omnifusion_phase11_best.pt"
trainer.train(save_path=save_path)

# 7. Plot Visualization
trainer.plot_final_results()